In [116]:
import pandas as pd
import matplotlib.pyplot as plt
import ydata_profiling
from datetime import datetime, timedelta

In [117]:
df = pd.read_csv('datasets/data.csv')

df.head(3)

,No,Water Meter ID,Reading ID,Reading Value,Reading Date,Previous Reading Value,Previous Reading Date,Reading Frequency,Reader ID,Type of Contract,Reading Validity,Certification on the ERP,Final Billing,Reason for Reading
0,1,IT-WM-001,READ-001,145.50,15/01/2024,120.30,15/12/2023,Monthly,READER-01,Residential,Valid,Yes,125.40,Routine
1,2,IT-WM-002,READ-002,89.75,16/01/2024,75.20,16/12/2023,Monthly,READER-02,Commercial,Valid,Yes,89.75,Routine
2,3,IT-WM-003,READ-003,NaN,17/01/2024,210.45,17/12/2023,Monthly,READER-03,Industrial,Invalid,No,0.00,Missing


Transformacion de variables

In [118]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501 entries, 0 to 500
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   No                        501 non-null    int64  
 1   Water Meter ID            501 non-null    object 
 2   Reading ID                501 non-null    object 
 3   Reading Value             494 non-null    float64
 4   Reading Date              501 non-null    object 
 5   Previous Reading Value    494 non-null    float64
 6   Previous Reading Date     496 non-null    object 
 7   Reading Frequency         501 non-null    object 
 8   Reader ID                 499 non-null    object 
 9   Type of Contract          500 non-null    object 
 10  Reading Validity          501 non-null    object 
 11  Certification on the ERP  500 non-null    object 
 12  Final Billing             484 non-null    float64
 13  Reason for Reading        478 non-null    object 
dtypes: float64

In [119]:
# dates transform
times = ['Reading Date', 'Previous Reading Date']

for i in times:
    df[i] = pd.to_datetime(df[i], errors='coerce')
    df[i] = df[i].dt.strftime('%Y-%m-%d')
    print(df[i].head(3))
    print('\n')

0    2024-01-15
1    2024-01-16
2    2024-01-17
Name: Reading Date, dtype: object


0    2023-12-15
1    2023-12-16
2    2023-12-17
Name: Previous Reading Date, dtype: object




/tmp/ipykernel_14684/3314024531.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[i] = pd.to_datetime(df[i], errors='coerce')
/tmp/ipykernel_14684/3314024531.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[i] = pd.to_datetime(df[i], errors='coerce')


In [120]:
# bool transform
# iba a agregar final billing pero el dataset no cumple con las condiciones del diccionario
# asi que lo voy a dejar como esta
bools = ['Certification on the ERP']
for i in bools:
    df[i] = df[i].str.lower()
    df[i] = df[i].replace({'true': True, 'false': False, '0': False, '1': True})
    df[i] = df[i].astype(bool)
    print(df[i].head(3))
    print('\n')

# vality
# Replace values in 'Reading Validity' column using the replace() method
df['Reading Validity'] = df['Reading Validity'].replace(
    {'Valid': True, 'Invalid': False, 'Válido': True, 'Inválido': False, 'Valido': True, 'Sospetto': False}
)
df['Reading Validity'] = df['Reading Validity'].astype(bool)
print(df['Reading Validity'].head(3))

0    True
1    True
2    True
Name: Certification on the ERP, dtype: bool


0     True
1     True
2    False
Name: Reading Validity, dtype: bool


Analisis de Variables nulas

In [121]:
null_count = {}

for col in df.columns:
    count = df[col].isnull().sum()
    if count > 0:
        null_count[col]=count
for col, count in null_count.items():
    print(col,":",count)


Reading Value : 7
Previous Reading Value : 7
Previous Reading Date : 13
Reader ID : 2
Type of Contract : 1
Final Billing : 17
Reason for Reading : 23


In [122]:
df.head(1)

,No,Water Meter ID,Reading ID,Reading Value,Reading Date,Previous Reading Value,Previous Reading Date,Reading Frequency,Reader ID,Type of Contract,Reading Validity,Certification on the ERP,Final Billing,Reason for Reading
0,1,IT-WM-001,READ-001,145.5,2024-01-15,120.3,2023-12-15,Monthly,READER-01,Residential,True,True,125.4,Routine


Pero antes si queremos estandarizar usando mediana y demas debemos establecer umbrales y demas

In [123]:
# reglas para analizar umbrales
validation_rules = {
    # numeric
    "Reading Value": {
        "type": "numeric", 
        "min": 0, 
        "max": 1000, 
        "allow_null": False, 
        "fill_strategy": "median"
    },
    "Previous Reading Value": {
        "type": "numeric",
        "min": 0,
        "max": 1000,
        "allow_null": True,
        "fill_strategy":  "median"
    },
    "Final Billing":{
        "type": "numeric",
        "min": 0.0,
        "max": 500.0,
        "allow_null": False,
        "fill_strategy":  "median"
    },
    # categorical
    "Type of Contract": {
        "type": "categorical",
        "allowed_values": ["residential", "comercial", "industrial"], 
        "allow_null": False,
        "fill_strategy": "mode",
        "mapping": {
            # agregar los de diferentes idiomas
            "Residential": "residential",  # Fix case sensitivity
            "residencial": "residential",
            "residenziale": "residential",
            "commerciale": "comercial",
            "industriale": "industrial",
        }
    },
    "Reading Validity":{
        "type": "categorical",
        "allowed_values": [True, False],  # Change to boolean values
        "allow_null": False,
        "fill_strategy": "mode",
        "mapping": {
            # agregar los de diferentes idiomas
            "valid": True,
            "invalid": False,
            "suspicious": False,
            "valido": True,
            "invalido": False,
            "sospechoso": False,
            "non valido": False,
            "sospetto": False,
        }
    },
    "Certification on the ERP":{
        "type": "categorical",
        "allowed_values": [True, False],  # Change to boolean values
        "allow_null": False,
        "fill_strategy": "mode",
        "mapping": {
            # agregar los de diferentes idiomas
            "yes": True,
            "no": False,
            "si": True,
            "y": True,
            "1": True,
            "0": False,
        }
    },
    "Reading Frequency": {
        "type": "categorical", 
        "allowed_values": ["monthly", "bimonthly"],
        "allow_null": True,
        "fill_strategy": "mode",
        "mapping": {
            "Monthly": "monthly",  # Fix case sensitivity
            "Bimonthly": "bimonthly",  # Fix case sensitivity
            "mensual": "monthly",
            "bimestral": "bimonthly",
            "mensile": "monthly",
            "bimestrale": "bimonthly",
        }
    },
}

In [124]:
def validate_numeric(df, column, rules):
    errors = {}
    # out of range values
    out_of_range = ((df[column] < rules["min"]) | (df[column] > rules["max"])).sum()
    if out_of_range > 0:
        df[column] = df[column].clip(lower=rules["min"], upper=rules["max"])
    errors["out_of_range_n"] = out_of_range


    # null operation
    null_before = df[column].isnull().sum()
    if null_before > 0 and not rules["allow_null"]:
        if rules["fill_strategy"] == "median":
            df[column] = df[column].fillna(df[column].median())
        elif rules["fill_strategy"] == "mode":
            df[column] = df[column].fillna(df[column].mode()[0])
    errors["null_values_n"] = null_before
    return df, errors

In [125]:
def validate_categoric(df, column, rules):
    errors = {}
    # using the map
    if "mapping" in rules:
        df[column] = df[column].map(rules["mapping"]).fillna(df[column])

    # null values
    null_before = df[column].isnull().sum()
    if null_before > 0 and not rules["allow_null"]:
        df[column] = df[column].fillna("NaN")
    errors["null_values_c"]=null_before

    # validate against allowed values
    invalid_values = ~df[column].isin(rules["allowed_values"])
    invalid_count = invalid_values.sum()
    
    if invalid_count > 0:
        # Replace invalid values with mode or first allowed value
        mode_val = df[column].mode()
        replacement = mode_val[0] if not mode_val.empty else rules["allowed_values"][0]
        df.loc[invalid_values, column] = replacement
        
    errors["invalid_values_c"] = invalid_count
    return df, errors

In [126]:
def transform_df(df,rules):
    error_report = {}
    for column, rule in rules.items():
        if column not in df.columns:
            continue
        if rule["type"] == "numeric":
            df, error = validate_numeric(df, column, rule)
            error_report[column] = error
        elif rule["type"] == "categorical":
            df, error = validate_categoric(df, column, rule)
            error_report[column] = error
    return df, error_report


In [127]:
pd.set_option('future.no_silent_downcasting', True) # error por el fill na

df_clean, report = transform_df(df, validation_rules)
print("Cleaning Succesfull!!")

for column, errors in report.items():
    print(column, errors)

Cleaning Succesfull!!
Reading Value {'out_of_range_n': np.int64(35), 'null_values_n': np.int64(7)}
Previous Reading Value {'out_of_range_n': np.int64(5), 'null_values_n': np.int64(7)}
Final Billing {'out_of_range_n': np.int64(150), 'null_values_n': np.int64(17)}
Type of Contract {'null_values_c': np.int64(1), 'invalid_values_c': np.int64(275)}
Reading Validity {'null_values_c': np.int64(0), 'invalid_values_c': np.int64(0)}
Certification on the ERP {'null_values_c': np.int64(0), 'invalid_values_c': np.int64(0)}
Reading Frequency {'null_values_c': np.int64(0), 'invalid_values_c': np.int64(103)}


In [128]:
df_clean.head(10)

,No,Water Meter ID,Reading ID,Reading Value,Reading Date,Previous Reading Value,Previous Reading Date,Reading Frequency,Reader ID,Type of Contract,Reading Validity,Certification on the ERP,Final Billing,Reason for Reading
0,1,IT-WM-001,READ-001,145.50,2024-01-15,120.30,2023-12-15,monthly,READER-01,residential,True,True,125.40,Routine
1,2,IT-WM-002,READ-002,89.75,2024-01-16,75.20,2023-12-16,monthly,READER-02,residential,True,True,89.75,Routine
2,3,IT-WM-003,READ-003,456.78,2024-01-17,210.45,2023-12-17,monthly,READER-03,residential,False,True,0.00,Missing
3,4,IT-WM-004,READ-004,320.10,2024-01-18,295.80,2023-12-18,bimonthly,READER-01,residential,True,True,156.30,Routine
4,5,IT-WM-001,READ-005,170.25,2024-02-15,145.50,2024-01-15,monthly,READER-02,residential,True,True,145.50,Routine
5,6,IT-WM-005,READ-006,55.30,2024-01-19,NaN,NaN,monthly,READER-04,residential,True,True,55.30,First Reading
6,7,IT-WM-006,READ-007,478.90,2024-01-20,450.25,2023-11-20,bimonthly,READER-03,residential,True,True,125.45,Routine
7,8,IT-WM-007,READ-008,123.45,2024-01-21,110.75,2023-12-21,monthly,READER-01,residential,True,True,98.70,Routine
8,9,IT-WM-008,READ-009,987.65,2024-01-22,890.40,2023-12-22,monthly,READER-05,residential,True,True,450.80,Suspected Leak
9,10,IT-WM-002,READ-010,95.20,2024-02-16,89.75,2024-01-16,monthly,READER-02,residential,True,True,95.20,Routine


In [129]:
df_clean.to_csv("datasets/cleanData.csv", index=False)